# FedDDI-GNN — Federated Drug-Drug Interaction Prediction (Colab)

This notebook runs the full FedDrift experimental pipeline end-to-end on a Colab GPU.

### Research direction
Every FL client trains on the **same shared global DDI graph** (DrugBank / OGB-DDI —
publicly available pharmacological knowledge). Privacy and heterogeneity come entirely
from each hospital's **private local clinical observations** — which interactions they
observe most frequently — not from graph ownership.

| What every client shares | What each client keeps private |
|---|---|
| Global DDI graph (message-passing topology) | Local observation distribution (ATC-class mix) |
| Fixed node features (structural, computed once) | Prescription frequencies / adverse event patterns |
| GraphSAGE architecture | Its own model weights and optimizer |

Heterogeneity is generated via **ATC-aware Dirichlet sampling** over training
interactions (not over graph nodes). Concept drift changes a client's sampling
distribution mid-run without ever touching the graph.

**Before running:** go to **Runtime → Change runtime type → T4 GPU**.

## 1. Clone the project from GitHub

In [ ]:
from google.colab import userdata
import os

token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/sid-zyy/fedddi.git
%cd fedddi
print('Working directory:', os.getcwd())

## 2. Install dependencies

In [9]:
%pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## 3. Patch OGB / verify environment

`colab_setup.py` patches a PyTorch ≥ 2.6 `torch.load` compatibility issue in
OGB's dataset loader and verifies GPU availability and required imports.

In [ ]:
!python colab_setup.py

## 4. Download the dataset

First call to `load_dataset()` triggers the OGB download of `ogbl-ddi`
(4,267 drug nodes, ~1.07 M training interactions). Downloads ~300 MB; takes
about a minute on a fresh Colab session.

In [ ]:
!python -c "from src.data.dataset import load_dataset; load_dataset(root='./data')"

## 5. Run the unit tests

101 tests covering the full pipeline — observations, drift simulator, encoder,
predictor, client construction, and training loop. All should pass in ~3 seconds
without needing the real dataset. If anything fails here, stop and investigate
before running the heavier cells below.

In [ ]:
!python -m pytest tests/ --ignore=tests/test_fl_aggregation.py -q

## 6. (Optional) Prepare ATC mapping

The ATC mapping connects each drug node to its Level-1 therapeutic class
(Cardiovascular, Oncology, Antibiotics, etc.). It is used by the Dirichlet
observation sampler to generate realistic per-hospital specialization patterns.

**Without it:** the simulator falls back to class-agnostic sampling — clients
still get heterogeneous observation sets, just not along therapeutic-area lines.
All Step 4 and Step 5 code runs correctly either way.

**With it:** the log will show messages like
`[Client 0] local observations: ... top_classes=[class 2: 55%, class 0: 30%, ...]`
which map to real ATC categories.

In [ ]:
!python scripts/prepare_atc_mapping.py

## 7. Centralized baseline — smoke test (learned embeddings)

Quick sanity check with the standard ogbl-ddi setup: learned node embeddings
(not the fixed structural features). 50 epochs, 1 run — just to confirm the
encoder + predictor are learning. You should see non-trivial, increasing Hits@20.

**Note:** This is the "upper bound" number (~0.876 reported on this codebase with
500 epochs), but it uses learned embeddings so it's NOT directly comparable to
the federated runs below, which use fixed structural features. See step 7b for
the matched baseline.

In [ ]:
!python train.py training.num_runs=1 training.epochs=50 wandb.mode=disabled device=cuda

## 7b. Centralized baseline — matched mode (fixed features)

**This is the number for the paper's comparison table.** Same architecture and
fixed features as the federated runs (Strategy 1, compute_fixed_global_features).
50 epochs smoke test first, then scale to 500 for the final number.

The gap between this and the federated results in steps 10–11 is the true
federation overhead (not the representation difference from learned embeddings).

**Smoke test (50 epochs, 1 run):**

In [ ]:
!python train.py \
    baseline.use_fixed_features=true \
    training.num_runs=1 \
    training.epochs=50 \
    wandb.mode=disabled \
    device=cuda

## 7c. Full centralized baseline — matched mode (500 epochs)

The final number. Expect ~2–4 hours on T4. This is the ceiling for federated
performance on fixed features — the gap between this and steps 10 and 11 is the
true federation cost.

In [ ]:
!python train.py \
    baseline.use_fixed_features=true \
    training.num_runs=5 \
    training.epochs=500 \
    wandb.mode=disabled \
    device=cuda \
    2>&1 | tee centralized_fixed_features.log

## 8. FL smoke test — stable FedAvg baseline (2 rounds)

2 rounds, 1 local epoch, 5 clients, **no drift** (`drift.events` is empty by
default). Every client gets its own Dirichlet-sampled observation set from the
shared global graph, runs GraphSAGE locally, and sends gradients to the server.

What to watch for in the logs:
- `[Client N] local observations: ... top_classes=[...]` — each client's
  therapeutic-area mix (different across clients = heterogeneity working).
- `Round X — server_eval Val Hits@20: ...` — server-side global evaluation.
- `Round X — federated Val Hits@20: ...` — weighted average across clients.

Both metric lines should be non-zero and in the same ballpark from round 1.

In [ ]:
!python main_fl.py \
    num_rounds=2 \
    local_epochs=1 \
    num_clients=5 \
    training.epochs=1 \
    device=cuda \
    wandb.mode=disabled

## 9. FL smoke test — with concept drift (sudden, 2 rounds)

Same 2-round smoke test, but with a sudden drift event on client 0 at round 1.
This exercises the full Step 5 path: the `ConceptDriftSimulator` fires,
re-samples client 0's observations with the new mixture, and you should see
a log line like:

```
[Round 1] Client 0 drift → [class 1: 80%, class 0: 10%, class 2: 5%, ...]
```

The graph is never modified — only client 0's training observations change.

**What the target_mixture keys mean** — they are ATC Level-1 class indices
(integers). Without the ATC mapping from step 6 all edges belong to class 0;
with the mapping each index maps to a therapeutic area (Cardiovascular,
Oncology, etc.).

In [ ]:
!python main_fl.py \
    num_rounds=2 \
    local_epochs=1 \
    num_clients=5 \
    training.epochs=1 \
    device=cuda \
    wandb.mode=disabled \
    'drift.events=[{client_id: 0, start_round: 1, kind: sudden, target_mixture: {0: 0.10, 1: 0.80, 2: 0.05, 3: 0.05}}]'

## 10. Full FL run — stable FedAvg baseline (100 rounds)

The headline federated baseline result: 100 rounds × 10 local epochs per client,
all 5 clients, **no drift**. This establishes the pre-drift plateau (mean of the
last 5 rounds before drift). Compare this against the centralized matched
baseline from step 7c to quantify federation overhead.

**Expected runtime:** ~4–5 hours on a T4. To preserve output across potential
session timeouts, redirect to a log file (see the cell below for how to
monitor and download it).

In [ ]:
!python main_fl.py \
    num_rounds=100 \
    local_epochs=10 \
    num_clients=5 \
    device=cuda \
    wandb.mode=disabled \
    2>&1 | tee fl_baseline.log

## 11. Full FL run — with concept drift (100 rounds)

The FedDrift scenario from the research proposal:
- **Rounds 1–39:** stable non-IID baseline (heterogeneous observations, no drift).
- **Round 40:** Client 0 drifts suddenly — simulating a new drug class entering
  clinical practice or a guideline change.
- **Rounds 50–60:** Client 1 drifts gradually over 10 rounds — simulating a
  slower demographic shift in patient populations.
- **Clients 2–4:** stable throughout.
- **Rounds 40–100:** recovery window. The system should re-adapt and return
  toward the pre-drift plateau.

Watch for the `[Round N] Client X drift →` log lines at rounds 40 and 50–60.
The code will also log the drift-specific metrics from proposal Section 11:
- `drift/plateau_hits20` — mean Hits@20 from last 5 rounds before drift (frozen at round 40).
- `drift/nadir_hits20` — lowest Hits@20 seen after drift (updated live).
- `drift/degradation` — plateau minus nadir. This is the key "how much did
  drift hurt" metric from the proposal.
- `drift/recovery_round` and `drift/rounds_to_recovery` — when the model
  returns to 95% of pre-drift plateau.

The graph is **never** modified. Only the observation distributions change.

**Expected runtime:** ~4–5 hours on a T4.

In [ ]:
!python main_fl.py \
    num_rounds=100 \
    local_epochs=10 \
    num_clients=5 \
    device=cuda \
    wandb.mode=disabled \
    'drift.events=[
      {client_id: 0, start_round: 40, kind: sudden,  target_mixture: {0: 0.40, 1: 0.35, 2: 0.25}},
      {client_id: 1, start_round: 50, kind: gradual, transition_rounds: 10, target_mixture: {2: 0.70, 0: 0.30}}
    ]' \
    2>&1 | tee fl_drift.log

## 12. Reading the results

**Metrics to watch per round (all runs):**
- `Round X — server_eval Val Hits@20: Y` — the server's global encoder
  evaluated on the complete val set. Should increase round-over-round in early
  rounds and plateau by round 35–40.
- `Round X — federated Val Hits@20: Y` — weighted average of per-client
  evaluations. Should track the server metric closely.

**Drift-specific metrics (step 11 only, logged every round after drift fires):**
- `drift/plateau_hits20` — set once at the first post-drift round. Mean of the
  5 stable rounds immediately before drift. This is the pre-drift baseline.
- `drift/nadir_hits20` — updated live as the model trains. The lowest Hits@20
  seen after drift. This is the maximum degradation.
- `drift/degradation` — plateau_hits20 minus nadir_hits20. The key metric:
  **how much did drift hurt model accuracy?**
- `drift/recovery_round` and `drift/rounds_to_recovery` — once the model
  returns to 95% of plateau, these are set and logged. Shows **how fast did
  the model re-adapt?**

**Log markers (drift runs only):**
- `ConceptDriftSimulator initialised — N client(s), M drift event(s)` —
  printed once at startup. Confirms drift events are active.
- `[Round N] Client X drift → [class A: P%, ...]` — printed when a client's
  mixture changes in that round. Shows the exact new distribution.
- `Drift plateau frozen at round N: Hits@20 = Y` — the pre-drift plateau is
  locked in once the window before drift is complete.
- `new post-drift nadir` — every time a worse Hits@20 is seen after drift.
- `RECOVERED to 95% of plateau` — first recovery message, with
  `rounds_to_recovery` showing the count.

**Summary numbers (end of each run):**
- `Best federated Val Hits@20 = Y at round R` — the peak FL performance.

**Paper comparison table template:**

| Setting | Val Hits@20 (mean ± std) | Notes |
|---|---|---|
| Centralized — learned embeddings (step 7c, full) | 0.876 ± 0.028 | Upper bound (not comparable to FL) |
| Centralized — fixed features (step 7c) | TBD ± TBD | Matched baseline for FedAvg |
| FedAvg (step 10, 100 rounds, no drift) | TBD | Plateau = pre-drift baseline |
| FedAvg + drift (step 11, 100 rounds) | TBD | Nadir = lowest post-drift; Degradation = plateau − nadir |

**Key interpretation:**
- Gap(Centralized — fixed) vs Gap(FedAvg — fixed) = Federation overhead
- Gap(FedAvg stable) vs Gap(FedAvg + drift) = Cost of concept drift alone
- Recovery time = How fast FedDrift (eventually) re-adapts vs vanilla FedAvg

## 13. Download logs

Steps 7c, 10, and 11 redirect output to `.log` files. Download them here.

In [ ]:
from google.colab import files
import os

for log_file in ['centralized_fixed_features.log', 'fl_baseline.log', 'fl_drift.log']:
    if os.path.exists(log_file):
        files.download(log_file)
        print(f'Downloaded {log_file}')
    else:
        print(f'{log_file} not found — run the corresponding cell first.')